# HippoVoice — Colab Runner

**Runtime:** A100 GPU (Runtime → Change runtime type → A100)

Run cells top to bottom. Each section is independent once setup is done.

| Section | What it does | GPU needed |
|---------|-------------|------------|
| 0. Setup | Install deps, clone repo | No |
| 1. Sanity checks | Run CPU tests | No |
| 2. Load models | STT + LLM + TTS | Yes |
| 3. Memory core | Test store + retriever | No |
| 4. Signal/noise benchmark | Core research claim | No |
| 5. Voice pipeline | End-to-end audio test | Yes |
| 6. LoCoMo benchmark | Full evaluation | Yes |

## 0. Setup

In [ ]:
# Verify A100
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
print(result.stdout)
assert 'A100' in result.stdout or 'A10' in result.stdout, \
    'Not on A100/A10G — change runtime type to A100 before continuing'

In [ ]:
# Install dependencies
!pip install -q chromadb networkx sentence-transformers soundfile jiwer fastapi pytest pytest-mock

# NeMo for Canary-Qwen STT
!pip install -q Cython
!pip install -q nemo_toolkit[asr]

# Fish Speech TTS
!git clone -q https://github.com/fishaudio/fish-speech /content/fish-speech
!pip install -q -e /content/fish-speech

print('Dependencies installed.')

In [ ]:
# Clone HippoVoice repo
import os

REPO = 'https://github.com/shivansh193/hippovoice.git'
REPO_DIR = '/content/hippovoice'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO} {REPO_DIR}

os.chdir(REPO_DIR)
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Working directory: {os.getcwd()}')
!ls

## 1. Sanity Checks (CPU)

In [ ]:
# Run all CPU-safe tests — no GPU required
!pytest tests/test_smoke.py tests/test_scorer.py tests/test_extractor.py \
        tests/test_decay.py tests/test_context.py -v --tb=short

In [ ]:
# Quick scorer sanity check — proves the math is right before touching models
from memory.scorer import compute_salience
import math

fear_fresh  = compute_salience(1.0, {'label': 'fear',    'intensity': 0.95}, 0, 0)
neut_old    = compute_salience(1.0, {'label': 'neutral', 'intensity': 0.01}, 0, 45)
fear_old    = compute_salience(1.0, {'label': 'fear',    'intensity': 0.90}, 0, 45)

print(f'Fear (fresh):          {fear_fresh:.3f}  — expect > 3.0')
print(f'Neutral (45 turns):    {neut_old:.3f}  — expect < 0.25 (compress zone)')
print(f'Fear (45 turns):       {fear_old:.3f}  — expect > 0.25 (still active)')

assert fear_fresh > 3.0
assert neut_old < 0.25
assert fear_old > 0.25
print('\n✓ Salience math correct')

## 2. Load Models

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Load Canary-Qwen 2.5B STT (~5GB VRAM, ~3 min)
from stt.model import load_canary
print('Loading Canary-Qwen 2.5B...')
canary = load_canary()
print(f'STT loaded. VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# Load Qwen3-8B LLM (~16GB VRAM, ~4 min)
# Swap model_name to 'Qwen/Qwen3-4B' if VRAM is tight after STT
from llm.client import LLMClient
print('Loading Qwen3-8B...')
llm = LLMClient(model_name='Qwen/Qwen3-8B')
print(f'LLM loaded. VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# Load Fish S2 Pro TTS (~9GB VRAM)
# NOTE: Only load this when testing voice output — skip for text-only benchmarks
# to save VRAM for the LLM.
#
# Uncomment when ready:
# from tts.model import load_fish_tts
# print('Loading Fish S2 Pro...')
# fish = load_fish_tts()
# print(f'TTS loaded. VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')
print('TTS skipped for now (saves ~9GB VRAM during benchmarking)')

In [ ]:
# Smoke test: LLM generates a coherent response
resp = llm.generate(
    system='You are a helpful assistant. Answer in one sentence.',
    messages=[{'role': 'user', 'content': 'What is 2 + 2?'}],
    max_tokens=40
)
print(f'LLM response: {resp}')
assert '4' in resp
print('✓ LLM working')

In [ ]:
# Smoke test: STT transcribes a generated audio clip
import numpy as np
import soundfile as sf
import tempfile, os

# Generate a 3-second sine wave as a dummy audio file
sr = 16000
t = np.linspace(0, 3, sr * 3)
audio = (np.sin(2 * np.pi * 440 * t) * 0.3).astype(np.float32)
with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
    sf.write(f.name, audio, sr)
    dummy_path = f.name

from stt.transcribe import transcribe, transcribe_with_embedding
text = transcribe(canary, dummy_path)
print(f'STT on sine wave (expect garbage): "{text}"')

_, emb = transcribe_with_embedding(canary, dummy_path)
print(f'Embedding shape: {emb.shape}  — expect (1280,)')
assert emb.shape == (1280,)
assert not np.any(np.isnan(emb))
print('✓ STT + embedding extraction working')
os.unlink(dummy_path)

## 3. Memory Core Tests (with real sentence-transformers)

In [ ]:
# Run store + retriever tests (these need sentence-transformers, skipped locally)
!pytest tests/test_store.py tests/test_retriever.py -v --tb=short

In [ ]:
# Manual walkthrough of the full memory loop
from memory.store import HippoMemory
from memory.extractor import extract_turn
from memory.retriever import hippo_retrieve
import numpy as np

mem = HippoMemory()

turns = [
    'My dog Max got diagnosed with cancer last week. I am absolutely devastated.',
    'The weather was nice today, a bit cloudy.',
    'I had cereal for breakfast this morning.',
    'Max had his first chemo session today. I held him the whole time.',
    'I watched TV for a bit after dinner.',
]

for i, turn in enumerate(turns):
    dummy_emb = np.zeros(1280, dtype=np.float32)  # text-only: no prosody
    memories = extract_turn(turn, dummy_emb, llm)
    for m in memories:
        m.update({'base_weight': 1.0, 'recall_count': 0, 'turn_created': i})
        mem.add(m)
    print(f'Turn {i}: extracted {len(memories)} memories from: "{turn[:60]}..."')

print(f'\nTotal memories stored: {mem.count()}')

print('\n--- Retrieval: "tell me about Max" ---')
results = hippo_retrieve('tell me about Max', mem, mem.graph, current_turn=5, top_k=5)
for r in results:
    print(f'  [{r["current_salience"]:.3f}] {r["content"]}')

## 4. Signal/Noise Benchmark (Core Research Claim)

In [ ]:
# This is the main result: HippoVoice vs Naive RAG noise contamination rate
# No GPU needed — runs on CPU with LLM mocked for memory extraction
from pipeline import HippoVoicePipeline
from baselines.naive_rag import NaiveRAG
from benchmarks.signal_noise.run import run_signal_noise_benchmark

print('Running signal/noise benchmark...')
print('(Uses real LLM for memory extraction — takes ~5 min)\n')

hippo = HippoVoicePipeline(llm_client=llm, text_only=True)
naive = NaiveRAG()

hippo_result = run_signal_noise_benchmark(hippo, 'HippoVoice')
naive_result = run_signal_noise_benchmark(naive, 'NaiveRAG')

print('=' * 50)
print(f'HippoVoice  — noise rate: {hippo_result["noise_rate"]:.1%}  '
      f'(signal={hippo_result["signal_count"]}, noise={hippo_result["noise_count"]})')
print(f'Naive RAG   — noise rate: {naive_result["noise_rate"]:.1%}  '
      f'(signal={naive_result["signal_count"]}, noise={naive_result["noise_count"]})')
print('=' * 50)

# Paper claim: HippoVoice < 10% noise, Naive RAG > 40%
print(f'\nTarget: HippoVoice < 10% — {"✓ PASS" if hippo_result["noise_rate"] < 0.10 else "✗ FAIL"}')
print(f'Target: Naive RAG  > 40% — {"✓ PASS" if naive_result["noise_rate"]  > 0.40 else "✗ FAIL"}')

In [ ]:
# Save benchmark results
import json, datetime

results = {
    'timestamp': datetime.datetime.now().isoformat(),
    'hippovoice': {k: v for k, v in hippo_result.items() if k != 'retrieved'},
    'naive_rag':  {k: v for k, v in naive_result.items()  if k != 'retrieved'},
}

with open('/content/signal_noise_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Results saved to /content/signal_noise_results.json')
print(json.dumps(results, indent=2))

## 5. End-to-End Voice Pipeline Test

In [ ]:
# Load TTS now (previously skipped)
from tts.model import load_fish_tts
print(f'VRAM before TTS: {torch.cuda.memory_allocated() / 1e9:.1f} GB')
fish = load_fish_tts()
print(f'VRAM after TTS:  {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# Record a voice turn using Colab's microphone
# Run this cell to record 5 seconds of audio
from IPython.display import Javascript, Audio, display
from google.colab import output
import base64, tempfile

RECORD_JS = """
const sleep = ms => new Promise(r => setTimeout(r, ms));
const b64 = b => new Promise(r => { const fr = new FileReader(); fr.onload = () => r(fr.result); fr.readAsDataURL(b); });
const stream = await navigator.mediaDevices.getUserMedia({audio: true});
const rec = new MediaRecorder(stream);
const chunks = [];
rec.ondataavailable = e => chunks.push(e.data);
rec.start();
console.log('Recording 5 seconds...');
await sleep(5000);
rec.stop();
await sleep(500);
const blob = new Blob(chunks);
const data = await b64(blob);
google.colab.kernel.invokeFunction('notebook.save_audio', [data], {});
"""

AUDIO_PATH = '/content/user_input.wav'

def save_audio(data):
    import soundfile as sf
    import numpy as np
    audio_bytes = base64.b64decode(data.split(',')[1])
    with open('/content/raw_input.webm', 'wb') as f:
        f.write(audio_bytes)
    # Convert webm to wav using ffmpeg
    os.system(f'ffmpeg -i /content/raw_input.webm -ar 16000 -ac 1 {AUDIO_PATH} -y -loglevel quiet')
    print(f'Audio saved: {AUDIO_PATH}')

output.register_callback('notebook.save_audio', save_audio)
display(Javascript(RECORD_JS))
print('Speak now — recording for 5 seconds...')

In [ ]:
# Run the voice turn through the full pipeline
from pipeline import HippoVoicePipeline
from tts.synthesize import synthesize
from stt.transcribe import transcribe_with_embedding
from memory.extractor import extract_turn
from memory.retriever import hippo_retrieve
from llm.context import build_system_prompt, BASE_COMPANION_PROMPT
import numpy as np

# Run manually (pipeline.process_turn() needs all models loaded)
print('Step 1: STT...')
transcript, audio_emb = transcribe_with_embedding(canary, AUDIO_PATH)
print(f'  Transcript: "{transcript}"')
print(f'  Embedding std: {audio_emb.std():.3f}')

print('\nStep 2: Memory extraction...')
memories = extract_turn(transcript, audio_emb, llm)
for m in memories:
    print(f'  [{m["emotion"]["label"]} {m["emotion"]["intensity"]:.2f}] {m["content"]}')

print('\nStep 3: LLM generation...')
system = build_system_prompt([], BASE_COMPANION_PROMPT)
response_text = llm.generate(
    system=system,
    messages=[{'role': 'user', 'content': transcript}],
    max_tokens=150
)
print(f'  Response: "{response_text}"')

print('\nStep 4: TTS...')
OUTPUT_PATH = '/content/response.wav'
synthesize(fish, response_text, OUTPUT_PATH)
print('  Done. Playing response:')
display(Audio(OUTPUT_PATH, autoplay=True))

## 6. LoCoMo Benchmark

In [ ]:
# Install HuggingFace datasets for LoCoMo
!pip install -q datasets
print('datasets installed')

In [ ]:
# Run LoCoMo evaluation (~30 min with Qwen3-8B)
from benchmarks.locomo.evaluate import run_locomo
from pipeline import HippoVoicePipeline

print('Running LoCoMo benchmark (30 conversations)...')
print('This will take ~20-30 minutes.\n')

pipe = HippoVoicePipeline(llm_client=llm, text_only=True)
result = run_locomo(pipeline=pipe, num_conversations=30)

print('=' * 50)
print(f'LoCoMo accuracy: {result["accuracy"]:.1%}  ({result["correct"]}/{result["total"]})')
print(f'Mem0 baseline:   ~65%')
print(f'Supermemory:     ~80%')
print('=' * 50)
print(f'\nTarget: > 65% — {"✓ PASS" if result["accuracy"] > 0.65 else "✗ FAIL"}')

# Save
import json
with open('/content/locomo_results.json', 'w') as f:
    json.dump({'accuracy': result['accuracy'], 'correct': result['correct'],
               'total': result['total']}, f, indent=2)
print('Saved to /content/locomo_results.json')

## 7. Save All Results + Download

Run this after benchmarks complete to download all results to your machine.

In [ ]:
import json, datetime
from google.colab import files

all_results = {
    'timestamp': datetime.datetime.now().isoformat(),
    'signal_noise': results if 'results' in dir() else 'not run',
    'locomo': {'accuracy': result['accuracy']} if 'result' in dir() else 'not run',
}

with open('/content/hippovoice_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

files.download('/content/hippovoice_results.json')
print('Downloaded.')